In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd

from src.data import load_matches, load_elo_baseline
from src.elo import prepare_matches, run_elo_updates, update_elo
from src.simulation import predict_match, assign_thirds
from src.features import update_team_history, update_h2h
import matplotlib as plt
import numpy as np

In [2]:
#Load everything from earlier notebooks
draw_threshold = joblib.load('draw_threshold.joblib')
wc_model = joblib.load('world_cup_model.joblib')
history_dict = joblib.load('final_history.joblib')
h2h_dict = joblib.load('final_h2h.joblib')
country_elo = joblib.load('final_elo.joblib')
model_h = joblib.load('home_goals_model.joblib')
model_a = joblib.load('away_goals_model.joblib')
scaler = joblib.load('scaler.joblib')

#Make a dataframe containing all the games in the group stage
group_stage_matches = load_matches()
group_stage_matches = prepare_matches(group_stage_matches, "2026-06-10", "2026-06-28")

In [3]:
#Group stage simulation

wc_groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

squad_values = {
    "Mexico": 226, "South Korea": 154, "South Africa": 50, "Czech Republic": 200,
    "Canada": 230, "Switzerland": 360, "Qatar": 23, "Bosnia and Herzegovina": 80,
    "Brazil": 1100, "Morocco": 350, "Haiti": 10, "Scotland": 200,
    "United States": 310, "Paraguay": 100, "Australia": 40, "Turkey": 320,
    "Germany": 850, "Curaçao": 5, "Ivory Coast": 300, "Ecuador": 230,
    "Netherlands": 600, "Japan": 290, "Sweden": 210, "Tunisia": 45,
    "Belgium": 450, "Egypt": 130, "Iran": 50, "New Zealand": 20,
    "Spain": 1000, "Cape Verde": 30, "Saudi Arabia": 30, "Uruguay": 480,
    "France": 1200, "Senegal": 250, "Iraq": 10, "Norway": 450,
    "Argentina": 800, "Algeria": 180, "Austria": 240, "Jordan": 15,
    "Portugal": 1050, "DR Congo": 110, "Uzbekistan": 35, "Colombia": 280,
    "England": 1500, "Croatia": 300, "Ghana": 200, "Panama": 20
}

rows=[]
for group, teams in wc_groups.items():
    for team in teams:
        rows.append({"Group": group,
                     "Team": team,
                     "Points": 0,
                     "GD": 0,
                     'GF': 0,
                     "GA": 0})
        
group_stage_result = pd.DataFrame(rows) #Gives us the starting group stage
for match in group_stage_matches.itertuples(index = False):
    
    if match.home_team not in country_elo or pd.isna(country_elo[match.home_team]):
        country_elo[match.home_team] = 1500.0
    if match.away_team not in country_elo or pd.isna(country_elo[match.away_team]):
        country_elo[match.away_team] = 1500.0
    
    x = predict_match(match.home_team, match.away_team, wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[match.home_team], country_elo[match.away_team])
    #print(f"{match.home_team} vs {match.away_team} ({x.home_score}-{x.away_score}, result: {x.outcome}), Prob: {x.probability}, diff: {x.diff}")
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'Points'] += x.home_points
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'Points'] += x.away_points
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GD'] += x.home_score - x.away_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GD'] += x.away_score - x.home_score
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GF'] += x.home_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GF'] += x.away_score
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GA'] += x.away_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GA'] += x.home_score
    
    if x.outcome == "Home Win": S = 1
    elif x.outcome == "Away Win": S = 0
    else: S = 0.5
    new_away, new_home = update_elo(S, match.neutral, match.K_factor,x.home_score, x.away_score, 
                                    country_elo[match.away_team], 
                                    country_elo[match.home_team])
    
    country_elo[match.home_team] = new_home
    country_elo[match.away_team] = new_away
    update_team_history(match, x.home_score, x.away_score, history_dict)
    update_h2h(match, h2h_dict)
    
group_stage_result = group_stage_result.sort_values(['Group', 'Points', 'GD', 'GF'], ascending=[True, False, False, False]).reset_index(drop=True)    
print(group_stage_result) #Predicted group stage results




   Group                    Team  Points  GD  GF  GA
0      A             South Korea       7   2   4   2
1      A            South Africa       5   1   4   3
2      A                  Mexico       3  -1   1   2
3      A          Czech Republic       1  -2   1   3
4      B             Switzerland       9   3   3   0
5      B  Bosnia and Herzegovina       4   1   2   1
6      B                  Canada       4   0   2   2
7      B                   Qatar       0  -4   1   5
8      C                Scotland       5   1   4   3
9      C                 Morocco       4   0   4   4
10     C                  Brazil       3   0   3   3
11     C                   Haiti       2  -1   1   2
12     D                Paraguay       7   4   4   0
13     D                  Turkey       4   0   2   2
14     D           United States       2  -1   1   2
15     D               Australia       2  -3   0   3
16     E                 Germany       7   2   3   1
17     E             Ivory Coast       7   2  

In [4]:
#Get teams that move on to round of 32

top2 = group_stage_result.groupby('Group').head(2).copy() #Teams that placed top 2 in their group which qualifies
third = group_stage_result.groupby('Group').nth(2).copy() #All teams that placed third (only 8 of them move on)

best8_third = third.sort_values(
    ['Points', 'GD', 'GF'], 
    ascending=[False, False, False]
).head(8)

thirds_slot_map = { #Needs to be fixed
    'E': 'ABCDEFGHIJKL',
    'I': 'ABCDEFGHIJKL', 
    'A': 'ABCDEFGHIJKL',
    'L': 'ABCDEFGHIJKL',
    'D': 'ABCDEFGHIJKL',
    'G': 'ABCDEFGHIJKL',
    'B': 'ABCDEFGHIJKL',
    'K': 'ABCDEFGHIJKL',
}

winners = {g: teams.iloc[0]['Team'] for g, teams in group_stage_result.groupby('Group')}
runners = {g: teams.iloc[1]['Team'] for g, teams in group_stage_result.groupby('Group')}

available_thirds = {row.Group: row.Team for row in best8_third.itertuples()}
assignments = {}
used = set()
for winner_group, eligible_groups in thirds_slot_map.items():
    for g in eligible_groups:
        if g in available_thirds and g not in used:
            assignments[winner_group] = available_thirds[g]
            used.add(g)
            break

r32_matchups = {
    73: (runners['A'], runners['B']),
    74: (winners['E'], assignments['E']),
    75: (winners['F'], runners['C']),
    76: (winners['C'], runners['F']),
    77: (winners['I'], assignments['I']),
    78: (runners['E'], runners['I']),
    79: (winners['A'], assignments['A']),
    80: (winners['L'], assignments['L']),
    81: (winners['D'], assignments['D']),
    82: (winners['G'], assignments['G']),
    83: (runners['K'], runners['L']),
    84: (winners['H'], runners['J']),
    85: (winners['B'], assignments['B']),
    86: (winners['J'], runners['H']),
    87: (winners['K'], assignments['K']),
    88: (runners['D'], runners['G'])
}

r32_rows = []
r16_teams = {}

for match, teams in r32_matchups.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])
    
    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        winner = teams[0] if x.diff > 0 else teams[1]

    r16_teams[match] = winner
    r32_rows.append({
        'home_team': teams[0],
        'away_team': teams[1],
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

    new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
                                    country_elo[teams[0]], 
                                    country_elo[teams[1]])
    
    country_elo[teams[0]] = new_home
    country_elo[teams[1]] = new_away


South Africa 2 - 3 Bosnia and Herzegovina → Bosnia and Herzegovina
Germany 0 - 1 Mexico → Mexico
Netherlands 2 - 1 Morocco → Netherlands
Scotland 1 - 2 Sweden → Sweden
France 1 - 0 Canada → France
Ivory Coast 1 - 0 Senegal → Ivory Coast
South Korea 0 - 1 Brazil → Brazil
Panama 0 - 0 Japan → Panama
Paraguay 0 - 0 Egypt → Paraguay
Iran 0 - 1 Norway → Norway
Portugal 1 - 1 England → Portugal
Uruguay 0 - 0 Austria → Uruguay
Switzerland 1 - 0 Algeria → Switzerland
Argentina 0 - 0 Spain → Argentina
Colombia 1 - 2 Croatia → Croatia
Turkey 0 - 0 Belgium → Turkey


In [5]:
#Round of 16
r16_matchups = {
    89: (r16_teams[74], r16_teams[77]),
    90: (r16_teams[73], r16_teams[75]),
    91: (r16_teams[76], r16_teams[78]),
    92: (r16_teams[79], r16_teams[80]),
    93: (r16_teams[83], r16_teams[84]),
    94: (r16_teams[81], r16_teams[82]),
    95: (r16_teams[86], r16_teams[88]),
    96: (r16_teams[85], r16_teams[87])
}

r16_rows = []
r8_teams = {}

for match, teams in r16_matchups.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])

    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        winner = teams[0] if x.diff > 0 else teams[1]

    r8_teams[match] = winner
    r16_rows.append({
        'home_team': teams[0],
        'away_team': teams[1],
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

    new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
                                    country_elo[teams[0]], 
                                    country_elo[teams[1]])
    
    country_elo[teams[0]] = new_home
    country_elo[teams[1]] = new_away


Mexico 1 - 1 France → Mexico
Bosnia and Herzegovina 2 - 2 Netherlands → Bosnia and Herzegovina
Sweden 0 - 1 Ivory Coast → Ivory Coast
Brazil 0 - 0 Panama → Brazil
Portugal 0 - 0 Uruguay → Portugal
Paraguay 0 - 0 Norway → Paraguay
Argentina 3 - 2 Turkey → Argentina
Switzerland 0 - 0 Croatia → Switzerland


In [6]:
#Quarter final
QF_matchups = {
  97: (r8_teams[89], r8_teams[90]),
  98: (r8_teams[93], r8_teams[94]), 
  99: (r8_teams[91], r8_teams[92]), 
  100: (r8_teams[95], r8_teams[96]), 
}

QF_rows = []
SF_teams = {}

for match, teams in QF_matchups.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])

    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        winner = teams[0] if x.diff > 0 else teams[1]

    SF_teams[match] = winner
    QF_rows.append({
        'home_team': teams[0],
        'away_team': teams[1],
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

    new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
                                    country_elo[teams[0]], 
                                    country_elo[teams[1]])
    
    country_elo[teams[0]] = new_home
    country_elo[teams[1]] = new_away


Mexico 0 - 0 Bosnia and Herzegovina → Mexico
Portugal 1 - 2 Paraguay → Paraguay
Ivory Coast 1 - 2 Brazil → Brazil
Argentina 0 - 0 Switzerland → Argentina


In [7]:
#Semi-Final

SF_matchups = {
  101: (SF_teams[97], SF_teams[98]),
  102: (SF_teams[99], SF_teams[100]) 
}

SF_rows = []
FINAL_teams = {}

for match, teams in SF_matchups.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])

    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        winner = teams[0] if x.diff > 0 else teams[1]

    FINAL_teams[match] = winner
    SF_rows.append({
        'home_team': teams[0],
        'away_team': teams[1],
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

    new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
                                    country_elo[teams[0]], 
                                    country_elo[teams[1]])
    
    country_elo[teams[0]] = new_home
    country_elo[teams[1]] = new_away

Mexico 1 - 1 Paraguay → Mexico
Brazil 2 - 0 Argentina → Brazil


In [8]:
for match, teams in {103: (FINAL_teams[101], FINAL_teams[102])}.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])

    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        winner = teams[0] if x.diff > 0 else teams[1]
        
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

print(x)



Mexico 0 - 1 Brazil → Brazil
Matchprediction(outcome='Away Win', home_points=0, away_points=3, probability=np.float64(0.4663), diff=np.float64(0.3286), home_score=0, away_score=1)


In [19]:
from collections import Counter
from src.simulation import run_tournament

results = []
for i in range(1000):
    w = run_tournament(wc_model, draw_threshold, history_dict, h2h_dict,
                       country_elo, model_h, model_a, scaler, group_stage_matches)
    results.append(w)

for team, count in Counter(results).most_common(10):
    print(f"{team}: {count:.1f}%")

Brazil: 89.0%
Ivory Coast: 73.0%
Germany: 73.0%
Spain: 65.0%
Ecuador: 64.0%
France: 63.0%
South Korea: 51.0%
Switzerland: 46.0%
Mexico: 38.0%
Morocco: 37.0%
